# Script 3.1 — Painel de Validação e Predições por Empresa

**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Roda imediatamente após o `03_cvm_modelagem.ipynb`. Consome os modelos
treinados e os artefatos de predição para gerar o painel completo de
resultados por empresa — a principal referência da banca para avaliar
o desempenho do pipeline.

| Bloco | Conteúdo |
|-------|----------|
| 1 | Tabela de validação hold-out 2024–2025: real vs. predito vs. erro por empresa |
| 2 | Evolução temporal completa: histórico + predição 2026 por empresa e setor |
| 3 | KPIs derivados das predições (Opção A — Penman 2013) com nota metodológica |
| 4 | Z\'\'-Score prospectivo calculado sobre os targets preditos |
| 5 | Painel comparativo intra-setor: todas as empresas lado a lado |

**Saídas:**
- `outputs/painel/` — todos os arquivos deste script
- `b31_validacao_holdout.csv`            — erro por empresa × target (hold-out)
- `b31_validacao_holdout_{setor}.png`    — real vs. predito visual
- `b31_predicao_2026_empresa.csv`        — predições 2026 por empresa × target
- `b31_kpis_derivados_2026.csv`          — KPIs calculados das predições
- `b31_zscore_prospectivo.csv`           — Z\'\' sobre predições 2026
- `b31_painel_setor_{setor}.png`         — comparativo intra-setor

## Etapa 0. Imports e Configuração

In [ ]:
import json, logging, pickle, warnings
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_PAINEL = PASTA_SAIDA / 'painel'
PASTA_PAINEL.mkdir(exist_ok=True)

logger = logging.getLogger('script_31')
logger.setLevel(logging.INFO)
logger.handlers.clear()
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'script_31.log', mode='w', encoding='utf-8')
_fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s'))
logger.addHandler(_fh)

# ── Constantes espelhadas do Script 3 ────────────────────────────────────────
_TARGET_BASES = ['DRE_3.01','DRE_3.11','EBITDA',
                 'BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2','DFC_MI_6.01']
_HORIZONTES   = ['_ITR_T1','_ITR_T2','_ITR_T3','_DFP']
_LOG_BASES     = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARCSINH_BASES = {'DFC_MI_6.01','DRE_3.11'}
LOG_TARGETS    = {f'TARGET_{b}{h}' for b in _LOG_BASES     for h in _HORIZONTES}
ARCSINH_TARGETS= {f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES}

NOME_VAR = {
    'DRE_3.01':'Receita Líquida','DRE_3.11':'Lucro Líquido','EBITDA':'EBITDA',
    'BPA_1':'Ativo Total','BPA_1.01':'Ativo Circulante','BPP_2.01':'Passivo Circulante',
    'BPP_2.03':'Patrimônio Líquido','BPP_2':'Passivo Total','DFC_MI_6.01':'FCO',
}
CORES_SETOR = {
    'Petróleo':'#1f4e79','Energia':'#2e75b6',
    'Varejo':'#ed7d31','Commodities':'#70ad47','Tecnologia':'#ffc000',
}

def inv_transform(y, target):
    y = np.asarray(y, float)
    if target in LOG_TARGETS:     return np.expm1(y)
    if target in ARCSINH_TARGETS: return np.sinh(y)
    return y

def smape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    d = (np.abs(yt)+np.abs(yp))/2.0
    m = d > 1e-9
    return float(np.mean(np.abs(yt[m]-yp[m])/d[m])) if m.sum()>0 else np.nan

logger.info("Script 3.1 iniciado em %s", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("✅ Etapa 0 concluída")

## Etapa 1. Carga dos Artefatos do Script 3

In [ ]:
with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'rb') as f:
    melhores = pickle.load(f)
with open(PASTA_SAIDA / 'selected_features_por_target.pkl', 'rb') as f:
    selected_features = pickle.load(f)

treino  = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste   = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_cvm_consolidado.parquet')

_p = PASTA_SAIDA / 'predicoes_teste_detalhadas.parquet'
df_pred_det = pd.read_parquet(_p) if _p.exists() else pd.DataFrame()
_p2 = PASTA_SAIDA / 'predicoes_prospectivas.parquet'
df_prosp    = pd.read_parquet(_p2) if _p2.exists() else pd.DataFrame()

TARGETS = [f'TARGET_{b}{h}' for b in _TARGET_BASES for h in _HORIZONTES
           if f'TARGET_{b}{h}' in melhores]

# Recria coluna SETOR se necessário
if 'SETOR' not in dataset.columns:
    setor_cols = [c for c in dataset.columns if c.startswith('setor_')]
    if setor_cols:
        dataset['SETOR'] = (dataset[setor_cols].idxmax(axis=1)
                            .str.replace('setor_',''))

# Mapas de lookup
mapa_nome  = {}; mapa_setor = {}; mapa_cnpj = {}
if 'CNPJ_CIA' in dataset.columns:
    _b = dataset.drop_duplicates('CNPJ_CIA')
    if 'NOME_CIA' in _b.columns:
        mapa_nome  = _b.set_index('CNPJ_CIA')['NOME_CIA'].to_dict()
        mapa_cnpj  = _b.set_index('NOME_CIA')['CNPJ_CIA'].to_dict()
    if 'SETOR' in _b.columns:
        mapa_setor = _b.set_index('CNPJ_CIA')['SETOR'].to_dict()

setores_empresas = {}
for cnpj, setor in mapa_setor.items():
    setores_empresas.setdefault(setor, []).append(cnpj)

print(f"Targets: {len(TARGETS)} | Empresas: {len(mapa_nome)} | Setores: {len(setores_empresas)}")
print(f"Predições hold-out: {len(df_pred_det):,} | Predições prospectivas: {len(df_prosp):,}")
logger.info("Artefatos carregados | targets=%d | empresas=%d", len(TARGETS), len(mapa_nome))

## Bloco 1. Tabela de Validação Hold-out 2024–2025

**Real vs. Predito por empresa, target e horizonte.**

Esta é a tabela principal para a banca: mostra o valor real observado
no hold-out, o valor predito pelo melhor modelo e os erros absoluto e
percentual simétrico (SMAPE). Valores reais existem → comparação direta.

*Referência: Hyndman & Athanasopoulos (2018) — avaliação em horizonte fixo.*

In [ ]:
print("\n" + "="*70)
print("  BLOCO 1 — Tabela de Validação Hold-out 2024–2025")
print("="*70)

rows_val = []

if not df_pred_det.empty:
    # Adiciona mapa de nome e setor
    df_pd = df_pred_det.copy()
    df_pd['NOME_CIA'] = df_pd['CNPJ_CIA'].map(mapa_nome)
    df_pd['SETOR']    = df_pd['CNPJ_CIA'].map(mapa_setor)

    # Filtra melhor modelo por target
    df_best = df_pd[
        df_pd.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
    ].copy()

    for _, row in df_best.iterrows():
        target = row['Target']
        transf = 'log1p' if target in LOG_TARGETS else ('arcsinh' if target in ARCSINH_TARGETS else 'none')
        yt = inv_transform([row['y_true']], target)[0]
        yp = inv_transform([row['y_pred']], target)[0]
        erro_abs = abs(yt - yp)
        denom    = (abs(yt) + abs(yp)) / 2.0
        smape_v  = float(erro_abs / denom) if denom > 1e-9 else np.nan
        base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
        hor  = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')
        rows_val.append({
            'empresa':   row.get('NOME_CIA',''),
            'setor':     row.get('SETOR',''),
            'target':    target,
            'variavel':  NOME_VAR.get(base, base),
            'horizonte': hor.replace('_',''),
            'algoritmo': row['Algoritmo'],
            'y_real_bi': round(yt/1e6, 4),
            'y_pred_bi': round(yp/1e6, 4),
            'erro_abs_bi': round(erro_abs/1e6, 4),
            'smape':     round(smape_v, 4) if not np.isnan(smape_v) else None,
            'dt_refer':  row.get('DT_REFER',''),
        })

    df_val = pd.DataFrame(rows_val).sort_values(['setor','empresa','variavel','horizonte'])
    df_val.to_csv(PASTA_PAINEL / 'b31_validacao_holdout.csv', index=False)
    print(f"  ✅ b31_validacao_holdout.csv ({len(df_val):,} linhas)")

    # Resumo por variável e horizonte
    print("\n  SMAPE médio por variável × horizonte (hold-out, escala original):")
    resumo = (df_val.groupby(['variavel','horizonte'])['smape']
              .agg(['mean','median','count'])
              .round(4))
    print(resumo.to_string())
else:
    df_val = pd.DataFrame()
    print("  ⚠️  predicoes_teste_detalhadas.parquet não encontrado.")

print("\n  ✅ Bloco 1 concluído")

### Bloco 1b. Figuras — Real vs. Predito por Setor

In [ ]:
# Um gráfico por setor × variável foco (Receita, Lucro, EBITDA)
# Scatter real vs. predito, colorido por empresa

BASES_FOCO = ['DRE_3.01','DRE_3.11','EBITDA']

if not df_val.empty:
    for setor in sorted(df_val['setor'].dropna().unique()):
        df_s = df_val[(df_val['setor'] == setor) &
                      (df_val['variavel'].isin([NOME_VAR[b] for b in BASES_FOCO]))]
        if df_s.empty:
            continue

        variaveis = [NOME_VAR[b] for b in BASES_FOCO if NOME_VAR[b] in df_s['variavel'].unique()]
        n_var = len(variaveis)
        fig, axes = plt.subplots(1, n_var, figsize=(5*n_var, 5), squeeze=False)
        empresas_s = sorted(df_s['empresa'].unique())
        cores_emp  = plt.cm.tab10(np.linspace(0, 1, len(empresas_s)))
        emp_cor    = {e: cores_emp[i] for i, e in enumerate(empresas_s)}

        for i_v, var in enumerate(variaveis):
            ax  = axes[0][i_v]
            df_v = df_s[df_s['variavel'] == var]
            for emp in empresas_s:
                sub = df_v[df_v['empresa'] == emp]
                if sub.empty: continue
                ax.scatter(sub['y_pred_bi'], sub['y_real_bi'],
                           color=emp_cor[emp], alpha=0.75, s=40,
                           label=emp[:12], zorder=3)
            # Linha identidade
            if not df_v.empty:
                lim_min = min(df_v['y_pred_bi'].min(), df_v['y_real_bi'].min())
                lim_max = max(df_v['y_pred_bi'].max(), df_v['y_real_bi'].max())
                ax.plot([lim_min, lim_max], [lim_min, lim_max],
                        'k--', lw=1.2, alpha=0.6, label='Identidade (y=x)')
            smape_med = df_v['smape'].mean()
            ax.set_title(f'{var}\nSMAPE médio = {smape_med:.1%}',
                         fontsize=9, fontweight='bold')
            ax.set_xlabel('Predito (R$ bi)', fontsize=8)
            ax.set_ylabel('Real (R$ bi)', fontsize=8)
            ax.legend(fontsize=6, loc='upper left', ncol=2)
            ax.grid(alpha=0.25)

        plt.suptitle(f'Validação Hold-out — {setor} (2024–2025)',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        fname = f'b31_validacao_holdout_{setor.replace(" ","_")}.png'
        plt.savefig(PASTA_PAINEL / fname, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ {fname}")

print("  ✅ Bloco 1b concluído")

## Bloco 2. Predições 2026 por Empresa

Aplica os melhores modelos sobre o último vetor de KPIs disponível
de cada empresa para gerar o valor predito no horizonte DFP 2026.

**Distinção metodológica (conforme seções 4.5.1 e 4.5.2 do TCC):**
A predição parte de dados *reais* observados (KPIs do ITR Q1/2026 ou
DFP 2025). Não é uma projeção hipotética — é a estimativa do modelo
para o próximo período com base no estado financeiro atual da empresa.

In [ ]:
print("\n" + "="*70)
print("  BLOCO 2 — Predições DFP 2026 por Empresa")
print("="*70)

rows_pred = []
TARGETS_DFP = [t for t in TARGETS if t.endswith('_DFP')]

# Último vetor de KPIs por empresa (do dataset consolidado)
def ultimos_kpis_empresa(cnpj, feats):
    df_e = dataset[dataset['CNPJ_CIA'] == cnpj]
    if df_e.empty: return None, None
    if 'ANO' in df_e.columns: df_e = df_e.sort_values('ANO')
    ul  = df_e.iloc[-1]
    ano = int(ul['ANO']) if 'ANO' in ul.index else None
    return ul, ano

for cnpj in mapa_nome:
    nome_emp = mapa_nome[cnpj]
    setor    = mapa_setor.get(cnpj, '')
    ul, ano_base = ultimos_kpis_empresa(cnpj, [])
    if ul is None: continue

    for target in TARGETS_DFP:
        if target not in melhores: continue
        alg_nome = melhores[target]
        feats    = selected_features.get(target, [])
        feats    = [f for f in feats if f in dataset.columns]
        if not feats: continue

        cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
        if not cam.exists(): continue
        try:
            obj    = joblib.load(cam)
            modelo = obj['modelo'] if isinstance(obj, dict) else obj
        except Exception: continue

        x = np.array([float(ul.get(f, 0) or 0) for f in feats]).reshape(1,-1)
        try:
            y_raw  = modelo.predict(x)
            y_pred = float(inv_transform(y_raw, target)[0])
        except Exception: continue

        base = target.replace('TARGET_','').replace('_DFP','')
        rows_pred.append({
            'cnpj': cnpj, 'empresa': nome_emp, 'setor': setor,
            'ano_base': ano_base, 'target': target,
            'variavel': NOME_VAR.get(base, base),
            'algoritmo': alg_nome,
            'y_pred_2026_bi': round(y_pred/1e6, 4),
        })

if rows_pred:
    df_pred26 = pd.DataFrame(rows_pred).sort_values(['setor','empresa','variavel'])
    df_pred26.to_csv(PASTA_PAINEL / 'b31_predicao_2026_empresa.csv', index=False)
    print(f"  ✅ b31_predicao_2026_empresa.csv ({len(df_pred26)} linhas, "
          f"{df_pred26['empresa'].nunique()} empresas)")

    # Resumo: Receita 2026 por empresa (fácil consulta para banca)
    rec = df_pred26[df_pred26['variavel']=='Receita Líquida'][
          ['empresa','setor','y_pred_2026_bi']].sort_values(['setor','y_pred_2026_bi'],
                                                             ascending=[True,False])
    print("\n  Receita Líquida predita DFP 2026 (R$ bilhões):")
    print(rec.to_string(index=False))
else:
    df_pred26 = pd.DataFrame()
    print("  ⚠️  Nenhuma predição gerada — verifique modelos em outputs/modelos/")

print("\n  ✅ Bloco 2 concluído")

## Bloco 3. KPIs Derivados das Predições 2026

Calcula indicadores financeiros a partir das predições dos targets,
seguindo a abordagem de projeção de demonstrativos de Penman (2013):
*"Financial Statement Analysis and Security Valuation"*, Cap. 15.

**Nota metodológica declarada (conforme boas práticas acadêmicas):**
Estes KPIs são derivados de predições, não de valores observados.
Propagam o erro das predições de forma multiplicativa.
Razões envolvendo duas predições independentes têm variância maior
do que razões com um denominador fixo. Devem ser interpretadas como
estimativas de ordem de grandeza, não como valores precisos.

In [ ]:
print("\n" + "="*70)
print("  BLOCO 3 — KPIs Derivados das Predições 2026")
print("  ⚠️  Nota: KPIs derivados propagam o erro das predições")
print("="*70)

rows_kpi_der = []

if not df_pred26.empty:
    # Pivota: empresa × variavel → valor predito
    pv = (df_pred26.pivot_table(index=['cnpj','empresa','setor','ano_base'],
                                  columns='variavel', values='y_pred_2026_bi')
                    .reset_index())
    pv.columns.name = None

    # Mapeamento de nomes legíveis para colunas
    REC = 'Receita Líquida'; LUC = 'Lucro Líquido'; EBI = 'EBITDA'
    AT  = 'Ativo Total';    AC  = 'Ativo Circulante'; PC = 'Passivo Circulante'
    PL  = 'Patrimônio Líquido'; PT = 'Passivo Total'; FCO = 'FCO'

    for _, row in pv.iterrows():
        def g(col):
            v = row.get(col)
            return float(v)*1e6 if (v is not None and not pd.isna(v)) else None

        rec=g(REC); luc=g(LUC); ebi=g(EBI)
        at=g(AT); ac=g(AC); pc=g(PC); pl=g(PL); pt=g(PT); fco=g(FCO)

        def ratio(num, den, scale=1):
            if num is None or den is None or abs(den) < 1e-9: return None
            return round(num/den*scale, 4)

        kpis_der = {
            'cnpj':    row['cnpj'],
            'empresa': row['empresa'],
            'setor':   row['setor'],
            'ano_pred': 2026,
            # Rentabilidade
            'margem_ebitda_pred':  ratio(ebi, rec),
            'margem_liquida_pred': ratio(luc, rec),
            'roe_pred':            ratio(luc, pl),
            'roa_pred':            ratio(luc, at),
            # Endividamento
            'endividamento_pred':  ratio(pt, at) if pt and at else ratio(pc, at),
            'alavancagem_de_pred': ratio(pt, pl) if pt and pl else None,
            # Liquidez (proxy — AC e PC preditos)
            'liquidez_corrente_pred': ratio(ac, pc),
            # Eficiência
            'giro_ativo_pred': ratio(rec, at),
            'fco_receita_pred': ratio(fco, rec),
            # Valores absolutos (bilhões)
            'receita_pred_bi':  round(rec/1e6,3) if rec else None,
            'lucro_pred_bi':    round(luc/1e6,3) if luc else None,
            'ebitda_pred_bi':   round(ebi/1e6,3) if ebi else None,
            # Flag de qualidade
            'nota_metodologica': 'KPIs derivados de predições ML — propagam erro das predições individuais',
        }
        rows_kpi_der.append(kpis_der)

    df_kpis_der = pd.DataFrame(rows_kpi_der).sort_values(['setor','empresa'])
    df_kpis_der.to_csv(PASTA_PAINEL / 'b31_kpis_derivados_2026.csv', index=False)
    print(f"  ✅ b31_kpis_derivados_2026.csv ({len(df_kpis_der)} empresas)")

    print("\n  Margem EBITDA predita 2026 por empresa:")
    _m = df_kpis_der[['empresa','setor','margem_ebitda_pred','margem_liquida_pred',
                        'roe_pred','endividamento_pred']].dropna(subset=['margem_ebitda_pred'])
    print(_m.sort_values(['setor','margem_ebitda_pred'],ascending=[True,False]).to_string(index=False))
else:
    df_kpis_der = pd.DataFrame()
    print("  ⚠️  Predições 2026 não disponíveis — Bloco 3 ignorado.")

print("\n  ✅ Bloco 3 concluído")

## Bloco 4. Z\'\'-Score Prospectivo (Altman Mercados Emergentes)

Calcula o Z\'\'  sobre os targets preditos para 2026, gerando
classificação de solvência prospectiva por empresa.

Esta é a contribuição metodológica específica do TCC: aplicar
o Z\'\'  sobre *predições* e não apenas sobre dados históricos.


In [ ]:
print("\n" + "="*70)
print("  BLOCO 4 — Z\'\'-Score Prospectivo 2026")
print("="*70)

ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10

def classificar_zona(z):
    if pd.isna(z):           return 'N/D'
    if z > ZONA_SEGURA:      return 'Segura'
    if z >= ZONA_CINZA_INF:  return 'Cinza'
    return 'Insolvência'

rows_z = []
if not df_kpis_der.empty:
    for _, row in df_kpis_der.iterrows():
        def g(col):
            v = row.get(col)
            return float(v)*1e6 if (v is not None and not pd.isna(v)) else None

        at=g('receita_pred_bi') ; # placeholder — busca AT real
        # Para Z\'\'  usa os valores de predições absolutas se disponíveis
        _pv_row = (df_pred26[(df_pred26['cnpj'] == row['cnpj'])]
                   .set_index('variavel')['y_pred_2026_bi']
                   .to_dict() if not df_pred26.empty else {})

        def gv(var):
            v = _pv_row.get(var)
            return float(v)*1e6 if (v is not None and not pd.isna(v)) else None

        at_v   = gv('Ativo Total')
        ac_v   = gv('Ativo Circulante')
        pc_v   = gv('Passivo Circulante')
        pl_v   = gv('Patrimônio Líquido')
        pt_v   = gv('Passivo Total')
        ebi_v  = gv('EBITDA')
        luc_v  = gv('Lucro Líquido')

        ebit_proxy = ebi_v * 0.85 if ebi_v else luc_v

        X1 = (ac_v-pc_v)/at_v  if (at_v and at_v>0 and ac_v and pc_v) else None
        X2 = pl_v/at_v          if (at_v and at_v>0 and pl_v)           else None
        X3 = ebit_proxy/at_v    if (at_v and at_v>0 and ebit_proxy)      else None
        X4 = pl_v/pt_v          if (pt_v and pt_v>0 and pl_v)            else None

        n_ok = sum(v is not None for v in [X1,X2,X3,X4])
        z_pp = None
        if n_ok >= 3:
            z_pp = 6.56*(X1 or 0)+3.26*(X2 or 0)+6.72*(X3 or 0)+1.05*(X4 or 0)

        rows_z.append({
            'empresa':    row['empresa'],
            'setor':      row['setor'],
            'ano_pred':   2026,
            'z_pp_pred':  round(z_pp, 4) if z_pp else None,
            'zona_pred':  classificar_zona(z_pp),
            'X1_CG_AT':   round(X1,4) if X1 else None,
            'X2_PL_AT':   round(X2,4) if X2 else None,
            'X3_EBIT_AT': round(X3,4) if X3 else None,
            'X4_PL_PT':   round(X4,4) if X4 else None,
            'componentes_ok': n_ok,
        })

    df_z_pred = pd.DataFrame(rows_z).sort_values(['setor','empresa'])
    df_z_pred.to_csv(PASTA_PAINEL / 'b31_zscore_prospectivo.csv', index=False)
    print(f"  ✅ b31_zscore_prospectivo.csv ({len(df_z_pred)} empresas)")

    print("\n  Z\'\'  prospectivo 2026 por empresa:")
    print(df_z_pred[['empresa','setor','z_pp_pred','zona_pred','componentes_ok']]
          .dropna(subset=['z_pp_pred'])
          .sort_values(['setor','z_pp_pred'],ascending=[True,False])
          .to_string(index=False))

    # Distribuição de zonas
    dist = df_z_pred['zona_pred'].value_counts()
    print("\n  Distribuição de zonas (2026 predito):")
    for zona, cnt in dist.items():
        pct = cnt/len(df_z_pred)*100
        print(f"    {zona:<15} {cnt} ({pct:.0f}%)")
else:
    df_z_pred = pd.DataFrame()
    print("  ⚠️  KPIs derivados não disponíveis — Bloco 4 ignorado.")

print("\n  ✅ Bloco 4 concluído")

## Bloco 5. Painel Comparativo Intra-Setor

Para cada setor: histórico 2015–2025 + predição 2026, todas as empresas,
nas três variáveis foco do TCC. Empresa com maior cobertura de dados
é destacada (linha mais espessa e anotada).

Esta figura é a síntese visual do pipeline completo —
dados reais + predição ML — diretamente usável na Seção 6 do TCC.

In [ ]:
print("\n" + "="*70)
print("  BLOCO 5 — Painel Comparativo Intra-Setor")
print("="*70)

if df_pred26.empty:
    print("  ⚠️  Predições 2026 não disponíveis — Bloco 5 ignorado.")
else:
    # Histórico anual (DFP) do dataset consolidado
    ds_dfp = dataset[dataset["ORIGEM"]=="DFP"].copy() if "ORIGEM" in dataset.columns else dataset.copy()
    ds_dfp["ANO"] = ds_dfp["ANO"].astype(int)

    BASES_FOCO = [("DRE_3.01","Receita Líquida"),
                  ("DRE_3.11","Lucro Líquido"),
                  ("EBITDA","EBITDA")]

    for setor, cnpjs in sorted(setores_empresas.items()):
        empresas_s = [mapa_nome[c] for c in cnpjs if c in mapa_nome]
        if not empresas_s: continue

        n_var = len(BASES_FOCO)
        fig, axes = plt.subplots(1, n_var, figsize=(5.5*n_var, 5.5), squeeze=False)
        cor_s  = CORES_SETOR.get(setor, "#3498db")
        cores_emp = plt.cm.tab10(np.linspace(0, 0.9, len(empresas_s)))
        emp_cor   = {e: cores_emp[i] for i,e in enumerate(empresas_s)}

        # Empresa destaque: maior nº de anos com DFP
        cob = {e: ds_dfp[ds_dfp["NOME_CIA"]==e]["ANO"].nunique() for e in empresas_s}
        emp_dest = max(cob, key=cob.get) if cob else None

        for i_b, (base, label_b) in enumerate(BASES_FOCO):
            ax = axes[0][i_b]

            for emp in empresas_s:
                cnpj_e = mapa_cnpj.get(emp)
                if not cnpj_e: continue

                # Histórico
                hist = ds_dfp[ds_dfp["NOME_CIA"]==emp][[" ANO" if " ANO" in ds_dfp.columns else "ANO", base]].dropna()
                hist.columns = ["ANO", base]
                hist = hist.sort_values("ANO")

                if hist.empty: continue
                destaque = emp == emp_dest
                lw  = 2.5 if destaque else 1.0
                ms  = 5   if destaque else 2
                alpha = 1.0 if destaque else 0.4
                zord  = 5   if destaque else 2

                ax.plot(hist["ANO"], hist[base]/1e6,
                        "o-", color=emp_cor[emp], lw=lw, ms=ms, alpha=alpha, zorder=zord,
                        label=emp[:12] if destaque else "_nolegend_")

                # Predição 2026
                pred_row = df_pred26[(df_pred26["cnpj"]==cnpj_e) &
                                     (df_pred26["variavel"]==label_b)]
                if not pred_row.empty:
                    y26 = float(pred_row["y_pred_2026_bi"].iloc[0])
                    ax.plot([hist["ANO"].iloc[-1], 2026],
                            [hist[base].iloc[-1]/1e6, y26],
                            "--", color=emp_cor[emp], lw=lw*0.9, alpha=alpha, zorder=zord)
                    ax.scatter([2026], [y26], color=emp_cor[emp],
                               s=50 if destaque else 20, zorder=zord+1,
                               marker="*" if destaque else "o")
                    if destaque:
                        ax.annotate(f"{y26:.1f} Bi",
                                    xy=(2026, y26), xytext=(4,6),
                                    textcoords="offset points",
                                    fontsize=7, color=emp_cor[emp], fontweight="bold")

            ax.axvspan(2020, 2021, alpha=0.08, color="gray")
            ax.axhline(0, color="#2c3e50", lw=0.6, ls="--", alpha=0.4)
            ax.set_title(f"{label_b}", fontsize=9, fontweight="bold")
            ax.set_xlabel("Ano", fontsize=8)
            ax.set_ylabel("R$ bilhões", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2)
            ax.legend(fontsize=7, loc="upper left")

        plt.suptitle(
            f"Setor: {setor} — Histórico DFP (2015–2025) + Predição 2026\n"
            f"Empresa destacada: {emp_dest or ''} (maior cobertura histórica)",
            fontsize=10, fontweight="bold"
        )
        plt.tight_layout()
        fname = f"b31_painel_setor_{setor.replace(' ','_')}.png"
        plt.savefig(PASTA_PAINEL / fname, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  ✅ {fname}")

print("\n  ✅ Bloco 5 concluído")

## Resumo Final

In [ ]:
n_figs = len(list(PASTA_PAINEL.glob("b31_*.png")))
n_csvs = len(list(PASTA_PAINEL.glob("b31_*.csv")))

print("\n" + "═"*70)
print("  RESUMO — Script 3.1")
print("═"*70)
print(f"  Empresas      : {len(mapa_nome)}")
print(f"  Targets DFP   : {len([t for t in TARGETS if t.endswith('_DFP')])}")
print(f"  CSVs gerados  : {n_csvs} em outputs/painel/")
print(f"  Figuras       : {n_figs} em outputs/painel/")
print("─"*70)
print("  Arquivos:")
for f in sorted(PASTA_PAINEL.iterdir()):
    print(f"    • {f.name}")
print("═"*70)
print("  ✅ Script 3.1 concluído — pronto para o Script 4")
logger.info("Script 3.1 concluído | csvs=%d | figs=%d", n_csvs, n_figs)